# Notebook 05 — Índices Espectrales Avanzados y LAI
**Teledetección — Maestría en Ingeniería | Universidad del Magdalena**  
**Sesión 3 | Viernes 24 Jul 2026**

En este notebook profundizamos en índices que van más allá del NDVI para detectar:
- **Contenido de clorofila** (NDRE — Red Edge)
- **Estrés hídrico en hoja** (NDMI — Moisture Index)
- **Vigor en doseles densos** (EVI — Enhanced Vegetation Index)
- **Área foliar** (LAI estimado)

**Contexto:** Los cultivos de cacao y banano del Norte del Magdalena son sensibles al estrés hídrico y a enfermedades foliares (Sigatoka Negra en banano). Estos índices detectan señales que el NDVI estándar no captura.

---

## Parte 0 — Instalación y autenticación

In [ ]:
!pip install geemap --quiet
import ee
import geemap
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.gridspec import GridSpec

ee.Authenticate()
ee.Initialize(project='tu-proyecto-gee')

## Parte 1 — Teoría: ¿por qué necesitamos más índices?

El NDVI tiene un problema fundamental: **se satura** cuando el dosel es muy denso (LAI > 3).

### Curva de saturación del NDVI
Simulemos esto con datos típicos de Sentinel-2:

In [ ]:
# Reflectancias típicas de Sentinel-2 según nivel de vegetación
# Formato: [B2-azul, B3-verde, B4-rojo, B5-RedEdge1, B8-NIR, B8A-RedEdge2, B11-SWIR1]
coberturas = {
    'Suelo desnudo':     [0.20, 0.22, 0.24, 0.25, 0.26, 0.25, 0.28],
    'Pastizal ralo':     [0.05, 0.09, 0.07, 0.14, 0.35, 0.34, 0.22],
    'Cacao joven':       [0.04, 0.07, 0.05, 0.12, 0.42, 0.41, 0.18],
    'Cacao adulto':      [0.03, 0.06, 0.04, 0.09, 0.48, 0.47, 0.14],
    'Banano sano':       [0.03, 0.07, 0.04, 0.11, 0.50, 0.49, 0.13],
    'Banano con Sigatoka': [0.04, 0.06, 0.06, 0.09, 0.42, 0.41, 0.20],
    'Bosque denso':      [0.02, 0.04, 0.02, 0.07, 0.51, 0.50, 0.10],
    'Agua':              [0.04, 0.04, 0.03, 0.02, 0.01, 0.01, 0.00],
}

# Índices
def calcular_indices(b):
    B2, B3, B4, B5, B8, B8A, B11 = b
    eps = 1e-6
    ndvi  = (B8 - B4)  / (B8  + B4  + eps)
    ndre  = (B8A - B5) / (B8A + B5  + eps)
    ndmi  = (B8A - B11)/ (B8A + B11 + eps)
    evi   = 2.5 * (B8 - B4) / (B8 + 6*B4 - 7.5*B2 + 1 + eps)
    savi  = 1.5 * (B8 - B4) / (B8 + B4 + 0.5 + eps)
    return dict(NDVI=ndvi, NDRE=ndre, NDMI=ndmi, EVI=evi, SAVI=savi)

print(f"{'Cobertura':<25} {'NDVI':>6} {'NDRE':>6} {'NDMI':>6} {'EVI':>6} {'SAVI':>6}")
print('-' * 55)
resultados = {}
for nombre, bandas in coberturas.items():
    idx = calcular_indices(bandas)
    resultados[nombre] = idx
    print(f"{nombre:<25} {idx['NDVI']:>6.3f} {idx['NDRE']:>6.3f} {idx['NDMI']:>6.3f} {idx['EVI']:>6.3f} {idx['SAVI']:>6.3f}")

In [ ]:
# Visualización: comparar todos los índices por cobertura
nombres = list(coberturas.keys())
indices_nombres = ['NDVI', 'NDRE', 'NDMI', 'EVI', 'SAVI']
colores = ['#2ecc71', '#e67e22', '#3498db', '#9b59b6', '#f1c40f']

x = np.arange(len(nombres))
ancho = 0.15

fig, ax = plt.subplots(figsize=(14, 6))
for i, (indice, color) in enumerate(zip(indices_nombres, colores)):
    valores = [resultados[n][indice] for n in nombres]
    ax.bar(x + i * ancho, valores, ancho, label=indice, color=color, alpha=0.85)

ax.set_xticks(x + 2 * ancho)
ax.set_xticklabels(nombres, rotation=30, ha='right', fontsize=10)
ax.set_ylabel('Valor del índice')
ax.set_title('Comparación de índices espectrales por tipo de cobertura\n(Norte del Magdalena — valores simulados Sentinel-2)')
ax.axhline(0, color='black', linewidth=0.5)
ax.legend()
ax.grid(axis='y', alpha=0.4)
plt.tight_layout()
plt.show()

print("\nClave de interpretación:")
print("- Banano con Sigatoka vs Banano sano: NDMI cae más que NDVI → detecta estrés hídrico/foliar")
print("- Cacao adulto vs Bosque denso: NDVI similar (saturado), pero NDRE y EVI los diferencian")

## Parte 2 — Red Edge (NDRE): la banda clave para cultivos

La **zona Red Edge** (680–750 nm) es donde la reflectancia vegetal da el salto más pronunciado: de absorber luz roja (clorofila) a reflejar NIR (estructura celular). La pendiente de ese salto codifica la **cantidad de clorofila**.

Sentinel-2 tiene tres bandas en esa zona: B5 (705 nm), B6 (740 nm), B7 (783 nm) a 20 m.

In [ ]:
# Curva espectral típica mostrando el Red Edge
# Longitudes de onda centrales de Sentinel-2
bandas_s2 = {
    'B2': 492, 'B3': 560, 'B4': 665,
    'B5': 705, 'B6': 740, 'B7': 783,
    'B8': 842, 'B8A': 865, 'B11': 1610, 'B12': 2190
}

# Reflectancias para 3 estados de vegetación
vegetacion_sana  = [0.03, 0.07, 0.04, 0.11, 0.25, 0.46, 0.48, 0.50, 0.13, 0.06]
vegetacion_media = [0.04, 0.07, 0.06, 0.09, 0.18, 0.38, 0.41, 0.42, 0.20, 0.10]
vegetacion_stress= [0.05, 0.06, 0.07, 0.08, 0.11, 0.29, 0.35, 0.37, 0.28, 0.16]

longitudes_onda = list(bandas_s2.values())

fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(longitudes_onda, vegetacion_sana,  'g-o', label='Banano sano (alta clorofila)', linewidth=2)
ax.plot(longitudes_onda, vegetacion_media, 'y-s', label='Cacao adulto (clorofila media)', linewidth=2)
ax.plot(longitudes_onda, vegetacion_stress,'r-^', label='Banano con Sigatoka (estrés)', linewidth=2, linestyle='--')

# Marcar zona Red Edge
ax.axvspan(680, 750, alpha=0.12, color='orange', label='Zona Red Edge (B5–B7)')
ax.axvspan(400, 480, alpha=0.08, color='blue', label='Azul (B2)')
ax.axvspan(550, 580, alpha=0.08, color='green')
ax.axvspan(630, 690, alpha=0.08, color='red')
ax.axvspan(830, 900, alpha=0.08, color='darkred', label='NIR (B8, B8A)')

# Etiquetas de bandas
for banda, wl in list(bandas_s2.items())[:8]:
    ax.axvline(wl, color='gray', linewidth=0.4, linestyle=':')
    ax.text(wl, 0.52, banda, rotation=90, fontsize=7, ha='center', color='gray')

ax.set_xlabel('Longitud de onda (nm)')
ax.set_ylabel('Reflectancia')
ax.set_title('Firmas espectrales Sentinel-2 — Norte del Magdalena\nEl Red Edge revela diferencias invisibles en el NDVI')
ax.set_xlim(450, 930)
ax.set_ylim(0, 0.58)
ax.legend(loc='upper left', fontsize=9)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Parte 3 — NDMI: detectar estrés hídrico en banano

El NDMI (Normalized Difference Moisture Index) usa el **SWIR1 (B11 = 1610 nm)** que es absorbido por el agua líquida en la hoja. Es el índice que usamos en la tesis de Bonivento (2024) para estimar película de agua libre.

```
NDMI = (B8A − B11) / (B8A + B11)
```

- NDMI > 0.2 → hoja hidratada (normal)
- NDMI 0 a 0.2 → estrés hídrico moderado
- NDMI < 0 → estrés severo o suelo desnudo

In [ ]:
# Zona bananera del Norte del Magdalena
zona_bananera = ee.Geometry.Rectangle([-74.35, 10.80, -74.10, 11.00])
zona_cacaotera = ee.Geometry.Rectangle([-74.20, 10.50, -73.90, 10.80])

def enmascarar_y_calcular_indices(imagen):
    scl = imagen.select('SCL')
    mascara = scl.neq(3).And(scl.neq(8)).And(scl.neq(9)).And(scl.neq(10))

    ndvi = imagen.normalizedDifference(['B8',  'B4' ]).rename('NDVI')
    ndre = imagen.normalizedDifference(['B8A', 'B5' ]).rename('NDRE')
    ndmi = imagen.normalizedDifference(['B8A', 'B11']).rename('NDMI')
    evi  = imagen.expression(
        '2.5 * (NIR - RED) / (NIR + 6*RED - 7.5*BLUE + 1)',
        {'NIR':  imagen.select('B8').divide(10000),
         'RED':  imagen.select('B4').divide(10000),
         'BLUE': imagen.select('B2').divide(10000)}
    ).rename('EVI')

    return imagen.addBands([ndvi, ndre, ndmi, evi]).updateMask(mascara)

coleccion = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
    .filterBounds(zona_bananera)
    .filterDate('2024-01-01', '2024-03-31')
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 15))
    .map(enmascarar_y_calcular_indices))

imagen_2024 = coleccion.median().clip(zona_bananera)
print(f"Imágenes en la colección: {coleccion.size().getInfo()}")
print("Bandas disponibles:", imagen_2024.bandNames().getInfo())

In [ ]:
# Mapa interactivo de los 4 índices
mapa = geemap.Map()
mapa.centerObject(zona_bananera, zoom=11)

vis_ndvi = {'bands': ['NDVI'], 'min': -0.1, 'max': 0.9, 'palette': ['#d73027','#f46d43','#fdae61','#ffffbf','#a6d96a','#66bd63','#1a9850']}
vis_ndre = {'bands': ['NDRE'], 'min': 0.0, 'max': 0.7, 'palette': ['#ffffcc','#c7e9b4','#7fcdbb','#41b6c4','#2c7fb8','#253494']}
vis_ndmi = {'bands': ['NDMI'], 'min': -0.2, 'max': 0.5, 'palette': ['#8c510a','#d8b365','#f6e8c3','#c7eae5','#5ab4ac','#01665e']}
vis_evi  = {'bands': ['EVI'],  'min': 0.0, 'max': 0.7, 'palette': ['#ffffe5','#f7fcb9','#d9f0a3','#addd8e','#78c679','#31a354','#006837']}

mapa.addLayer(imagen_2024, vis_ndvi, 'NDVI')
mapa.addLayer(imagen_2024, vis_ndre, 'NDRE (Red Edge)')
mapa.addLayer(imagen_2024, vis_ndmi, 'NDMI (Humedad hoja)')
mapa.addLayer(imagen_2024, vis_evi,  'EVI')
mapa.addLayer(imagen_2024, {'bands': ['B8','B4','B3'], 'min': 0, 'max': 5000, 'gamma': 1.4}, 'Falso Color NIR', False)

mapa

## Parte 4 — Estimación de LAI

El **LAI** (Leaf Area Index) no es un índice radiométrico directo sino una propiedad biofísica (m² de hoja por m² de suelo). Se estima por regresión desde NDVI o NDRE. La ecuación más usada para cultivos tropicales es:

```
LAI ≈ −ln((0.69 − NDVI) / 0.59) / 0.91    [Beer-Lambert simplificado]
```

O por regresión cuadrática sobre NDRE para mayor precisión en cultivos densos.

In [ ]:
# Estimar LAI desde NDVI (Beer-Lambert simplificado)
ndvi_img = imagen_2024.select('NDVI')

# Solo valores válidos de NDVI (0.1 a 0.9 para vegetación)
ndvi_valido = ndvi_img.updateMask(ndvi_img.gt(0.1).And(ndvi_img.lt(0.90)))

# Fórmula de Baret & Guyot (1991) — ampliamente usada en cultivos
lai = ndvi_valido.expression(
    '(-1.0 / 0.91) * log((0.69 - NDVI) / 0.59)',
    {'NDVI': ndvi_valido}
).rename('LAI').clamp(0, 8)

vis_lai = {'min': 0, 'max': 6, 'palette': ['#ffffcc','#d9f0a3','#78c679','#31a354','#006837','#00441b']}
mapa_lai = geemap.Map()
mapa_lai.centerObject(zona_bananera, zoom=11)
mapa_lai.addLayer(lai, vis_lai, 'LAI estimado (m²/m²)')
mapa_lai.addLayer(imagen_2024, {'bands': ['B4','B3','B2'], 'min': 0, 'max': 3000, 'gamma': 1.4}, 'Color natural', False)
mapa_lai

In [ ]:
# Estadísticas LAI por zona
stats_lai = lai.reduceRegion(
    reducer=ee.Reducer.mean().combine(ee.Reducer.stdDev(), sharedInputs=True)
                              .combine(ee.Reducer.min(),    sharedInputs=True)
                              .combine(ee.Reducer.max(),    sharedInputs=True),
    geometry=zona_bananera,
    scale=20,
    maxPixels=1e9
).getInfo()

print("=== LAI zona bananera Norte del Magdalena (Ene–Mar 2024) ===")
print(f"  Media:    {stats_lai.get('LAI_mean', 0):.2f} m²/m²")
print(f"  StdDev:   {stats_lai.get('LAI_stdDev', 0):.2f}")
print(f"  Mínimo:   {stats_lai.get('LAI_min', 0):.2f}")
print(f"  Máximo:   {stats_lai.get('LAI_max', 0):.2f}")
print()
print("Referencia — valores típicos de campo:")
print("  Banano adulto sano:   LAI 4–6 m²/m²")
print("  Cacao bajo sombra:    LAI 3–5 m²/m²")
print("  Pastizal:             LAI 1–3 m²/m²")
print("  Suelo con rastrojos:  LAI < 1 m²/m²")

## Parte 5 — Serie temporal de NDMI: detectar sequía en banano

Comparamos el NDMI en la zona bananera entre temporada seca (Ene–Mar) y lluviosa (Sep–Nov) para ver cómo el estrés hídrico varía en el año.

In [ ]:
# Comparación NDMI seca vs lluvia
periodos = [
    ('Seca 2023', '2023-01-01', '2023-03-31'),
    ('Lluvia 2023', '2023-09-01', '2023-11-30'),
    ('Seca 2024', '2024-01-01', '2024-03-31'),
    ('Lluvia 2024', '2024-09-01', '2024-11-30'),
]

def ndmi_periodo(nombre, f_inicio, f_fin):
    img = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
        .filterBounds(zona_bananera)
        .filterDate(f_inicio, f_fin)
        .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20))
        .map(enmascarar_y_calcular_indices)
        .median().clip(zona_bananera))

    stats = img.select('NDMI').reduceRegion(
        reducer=ee.Reducer.mean().combine(ee.Reducer.stdDev(), sharedInputs=True),
        geometry=zona_bananera,
        scale=20,
        maxPixels=1e9
    ).getInfo()

    return {'periodo': nombre, 'ndmi_media': stats.get('NDMI_mean', 0), 'ndmi_std': stats.get('NDMI_stdDev', 0)}

resultados_temporada = [ndmi_periodo(*p) for p in periodos]

# Gráfico
nombres_p = [r['periodo'] for r in resultados_temporada]
medias    = [r['ndmi_media'] for r in resultados_temporada]
stds      = [r['ndmi_std']   for r in resultados_temporada]
colores_p = ['#e74c3c', '#3498db', '#e67e22', '#2980b9']

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(nombres_p, medias, color=colores_p, alpha=0.8, width=0.5, yerr=stds, capsize=5)
ax.axhline(0.2, color='green', linestyle='--', linewidth=1.5, label='Umbral hidratación normal (0.2)')
ax.axhline(0.0, color='red',   linestyle='--', linewidth=1,   label='Umbral estrés severo (0)')

for bar, val in zip(bars, medias):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, f'{val:.3f}',
            ha='center', va='bottom', fontsize=10, fontweight='bold')

ax.set_ylabel('NDMI medio')
ax.set_title('NDMI estacional en zona bananera del Norte del Magdalena\nDetección de variación en contenido de agua foliar')
ax.legend()
ax.grid(axis='y', alpha=0.4)
ax.set_ylim(-0.1, 0.6)
plt.tight_layout()
plt.show()

## Parte 6 — Exportar índices a Google Drive

In [ ]:
# Exportar los 4 índices como un stack multiband
stack_indices = imagen_2024.select(['NDVI', 'NDRE', 'NDMI', 'EVI'])

tarea = ee.batch.Export.image.toDrive(
    image=stack_indices,
    description='indices_avanzados_bananera_2024',
    folder='Teledeteccion_S3',
    fileNamePrefix='indices_avanzados_bananera_2024',
    region=zona_bananera,
    scale=20,
    maxPixels=1e9,
    fileFormat='GeoTIFF'
)
tarea.start()
print("Exportación iniciada. Revisa: code.earthengine.google.com → Tasks")
print("El archivo aparecerá en tu Google Drive → carpeta Teledeteccion_S3")

---
## Resumen

| Índice | Qué detecta | Ventaja sobre NDVI |
|--------|-------------|--------------------|
| NDRE | Clorofila foliar | No se satura en doseles densos |
| NDMI | Agua en hoja | Detecta estrés hídrico antes que el NDVI |
| EVI | Vigor de vegetación | Corrige aerosoles y reflectancia de suelo |
| LAI | Área foliar total | Cuantifica biomasa; base para modelos de rendimiento |

**Conexión con la Sesión 4:** en el Notebook 07 usaremos NDRE + NDMI como variables de entrada para el clasificador supervisado.